# Latent-variable update following stimulation

**Goal.** Understand how a behavioral-model latent variable (e.g. `sumQ = Q_left + Q_right`)
is *updated* from trial to trial, and whether **stimulation** changes that update.

**Logic.**
- After a **reward** trial the chosen value goes up, so the latent typically moves one way;
  after a **no-reward** trial it moves the other way. That trial-to-trial change is the *update*.
- We align the latent trajectory to a **trigger trial** and follow it for several trials.
- Triggers are split into four groups: `reward/no-reward` x `stim/control`, so we can ask
  *"is the update following a stimulated reward (or no-reward) trial different from an
  un-stimulated one?"*

**Alignment note.** The model only updates on **responded** trials, so one "lag" here is one
responded (valid) trial. No-response trials are skipped. Because at least 4 non-stim trials
follow each stimulation, the post-stim windows are clean; we additionally require control
windows to contain no laser trial, so the two groups are directly comparable.


In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

print(f"env ready; modules from {MODULE_PATH}")


In [ ]:
# =============================================================================
# 2. CONFIG
# =============================================================================
# Behavior session (model fits + behavior NWB are keyed by this core name).
session_date = "863973_2026-08-25_14-18-29"

# Fitted model + latent to study. The latent is the variable whose *update*
# (trial-to-trial change) we analyze. "sumQ" (= Q_left + Q_right after the
# update) is the default; see behavior_utils.extract_fitted_data for the full
# list (e.g. "deltaQ", "chosenQ", "QL", "QR", "right_choice_probability", "RPE").
MODEL_ALIAS = "QLearning_L2F1_softmax"
LATENT_NAME = "sumQ"

# How to load the behavior NWB (must contain the laser_on_trial trials column):
#   False -> NWBUtils.read_behavior_nwb(session_date)  (same NWB the model was fit on)
#   True  -> NWBUtils.combine_nwb(ephys_session)       (fallback; set EPHYS_SESSION)
USE_COMBINED_NWB = False
EPHYS_SESSION = None  # only used if USE_COMBINED_NWB=True; None -> auto-discover

# Event-aligned window, in *valid-trial* steps relative to the trigger trial
# (lag 0 = the trigger). Negative lags are the pre-trigger baseline.
LAGS = [-2, -1, 0, 1, 2, 3, 4]
BASELINE_LAG = -1   # trajectories are expressed as y[lag] - y[BASELINE_LAG]

# Only keep triggers whose window contains no *other* laser trial, so stim and
# control windows are equally "clean" (the trigger itself may be a laser trial).
REQUIRE_CLEAN_WINDOW = True

RESULTS_DIR = Path(f"/root/capsule/scratch/latent_update_stim/{session_date}")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"session   : {session_date}")
print(f"model     : {MODEL_ALIAS}")
print(f"latent    : {LATENT_NAME}")
print(f"lags      : {LAGS} (baseline at {BASELINE_LAG})")
print(f"output dir: {RESULTS_DIR}")


## 3. Load behavior + latent, build valid-trial arrays

Load the behavior NWB (with `laser_on_trial`), restrict to **responded** trials, and build
four aligned arrays over valid trials: the latent `y`, the `reward` outcome (0/1), the `stim`
flag (0/1), and `abs_idx` (the absolute trial index of each valid trial). The latent length is
asserted to equal the number of valid trials so the fit and NWB are guaranteed aligned.


In [ ]:
# =============================================================================
# 3. LOAD BEHAVIOR + LATENT, BUILD VALID-TRIAL ARRAYS
# =============================================================================
from nwb_utils import NWBUtils
from behavior_utils import get_fitted_latent, extract_fitted_data

# --- behavior NWB (with laser_on_trial) ---
if USE_COMBINED_NWB:
    ephys_session = EPHYS_SESSION
    if ephys_session is None:
        cands = sorted(Path("/root/capsule/scratch/psth_results").glob(
            f"ecephys_{session_date}_sorted*_0.2s.zarr"))
        if not cands:
            raise FileNotFoundError("Could not auto-discover ephys_session; set EPHYS_SESSION.")
        ephys_session = cands[0].name[:-len("_0.2s.zarr")]
    nwb, _ = NWBUtils.combine_nwb(session_name=ephys_session)
else:
    nwb = NWBUtils.read_behavior_nwb(session_name=session_date)

colnames = list(nwb.trials.colnames)
if "laser_on_trial" not in colnames:
    raise KeyError(
        "No 'laser_on_trial' column in this NWB. Set USE_COMBINED_NWB=True. "
        f"(available columns: {colnames})")

# --- trials -> valid-trial (responded) arrays ---
responses = np.asarray(nwb.trials["animal_response"][:])
rewardedL = np.asarray(nwb.trials["rewarded_historyL"][:]).astype(bool)
rewardedR = np.asarray(nwb.trials["rewarded_historyR"][:]).astype(bool)
laser_on  = np.asarray(nwb.trials["laser_on_trial"][:]).astype(int)

valid_mask = responses != 2
abs_idx    = np.where(valid_mask)[0]              # absolute trial index per valid trial
reward     = (rewardedL | rewardedR).astype(int)[valid_mask]
stim       = laser_on[valid_mask]
n_valid    = int(valid_mask.sum())

# --- fitted latent (aligned to valid trials) ---
fit = get_fitted_latent(session_date, MODEL_ALIAS)
y = extract_fitted_data(nwb, fitted_latent=fit, model_alias=MODEL_ALIAS, latent_name=LATENT_NAME)
if y is None:
    raise ValueError(f"Could not extract latent '{LATENT_NAME}' from '{MODEL_ALIAS}'.")
y = np.asarray(y, dtype=float)

if len(y) != n_valid:
    raise ValueError(
        f"Latent length ({len(y)}) != number of valid trials ({n_valid}); "
        "the fit and this NWB may be misaligned.")

n_rew_stim   = int(((stim == 1) & (reward == 1)).sum())
n_norew_stim = int(((stim == 1) & (reward == 0)).sum())
print(f"all trials        : {len(responses)}")
print(f"valid (responded) : {n_valid}")
print(f"  rewarded        : {int(reward.sum())}")
print(f"  stim (laser)    : {int(stim.sum())}  (rewarded={n_rew_stim}, no-reward={n_norew_stim})")
print(f"latent '{LATENT_NAME}': min={np.nanmin(y):.3f}, max={np.nanmax(y):.3f}, mean={np.nanmean(y):.3f}")


## 4. What does an "update" look like?

Before adding stimulation, confirm the basic effect: the one-step update
`d[i] = y[i] - y[i-1]` should have opposite signs for **reward** vs **no-reward** trials
(computed on non-stim trials, i.e. the natural dynamics).


In [ ]:
# =============================================================================
# 4. WHAT DOES AN "UPDATE" LOOK LIKE?  (control/non-stim trials only)
# =============================================================================
d = np.full(n_valid, np.nan)
d[1:] = y[1:] - y[:-1]

ctrl = stim == 0
upd_rew   = d[ctrl & (reward == 1)]
upd_norew = d[ctrl & (reward == 0)]
upd_rew   = upd_rew[~np.isnan(upd_rew)]
upd_norew = upd_norew[~np.isnan(upd_norew)]

means = [np.mean(upd_rew), np.mean(upd_norew)]
sems  = [np.std(upd_rew, ddof=1) / np.sqrt(len(upd_rew)),
         np.std(upd_norew, ddof=1) / np.sqrt(len(upd_norew))]

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.bar([0, 1], means, yerr=sems, color=["#2ca02c", "#d62728"], alpha=0.85, capsize=5)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks([0, 1]); ax.set_xticklabels(["reward", "no-reward"])
ax.set_ylabel(f"one-step update  d{LATENT_NAME}  (y[i] - y[i-1])")
ax.set_title(f"Natural update (non-stim trials)\n{LATENT_NAME}")
u, p = mannwhitneyu(upd_rew, upd_norew, alternative="two-sided")
ax.text(0.5, 0.95, f"MWU p={p:.2g}", transform=ax.transAxes, ha="center", va="top")
plt.tight_layout()
fig.savefig(RESULTS_DIR / f"update_reward_vs_noreward_{LATENT_NAME}.png", dpi=150)
plt.show()
print(f"reward   : d{LATENT_NAME} = {means[0]:+.4f} +/- {sems[0]:.4f}  (n={len(upd_rew)})")
print(f"no-reward: d{LATENT_NAME} = {means[1]:+.4f} +/- {sems[1]:.4f}  (n={len(upd_norew)})")


## 5. Event-aligned latent trajectories

For every trigger trial we extract the latent at each lag, baseline-subtracted at
`BASELINE_LAG`, giving `y[lag] - y[baseline]`. Triggers are grouped by outcome
(`reward`/`no_reward`) and condition (`stim`/`control`). With `REQUIRE_CLEAN_WINDOW`,
only triggers whose window contains no *other* laser trial are kept.


In [ ]:
# =============================================================================
# 5. EVENT-ALIGNED LATENT TRAJECTORIES (reward/no-reward x stim/control)
# =============================================================================
LAGS = list(LAGS)
_nonzero = [l for l in LAGS if l != 0]

def _window_clean(i):
    for l in _nonzero:
        j = i + l
        if 0 <= j < n_valid and stim[j] == 1:
            return False
    return True

def build_trigger_rows(series):
    rows = []
    b0 = BASELINE_LAG
    for i in range(n_valid):
        b = i + b0
        if b < 0 or b >= n_valid or np.isnan(series[b]):
            continue
        if REQUIRE_CLEAN_WINDOW and not _window_clean(i):
            continue
        traj = np.full(len(LAGS), np.nan)
        for k, l in enumerate(LAGS):
            j = i + l
            if 0 <= j < n_valid:
                traj[k] = series[j] - series[b]
        rows.append(dict(
            i=i,
            outcome="reward" if reward[i] == 1 else "no_reward",
            cond="stim" if stim[i] == 1 else "control",
            traj=traj,
        ))
    return rows

GROUP_KEYS = [("reward", "stim"), ("reward", "control"),
              ("no_reward", "stim"), ("no_reward", "control")]

def group_trajectories(series):
    rows = build_trigger_rows(series)
    g = {}
    for key in GROUP_KEYS:
        sel = [r["traj"] for r in rows if (r["outcome"], r["cond"]) == key]
        g[key] = np.vstack(sel) if sel else np.empty((0, len(LAGS)))
    return g

groups = group_trajectories(y)
for key, M in groups.items():
    print(f"{key[0]:9s} / {key[1]:7s}: {M.shape[0]} triggers")


## 6. Trajectory figure: stim vs control, split by outcome

Mean +/- SEM latent trajectory following reward (left) and no-reward (right) triggers, comparing
`stim` (blue) vs `control` (grey). Stars mark lags where a Mann-Whitney U test separates stim
from control (p < 0.05).


In [ ]:
# =============================================================================
# 6. TRAJECTORY FIGURE: stim vs control, split by outcome
# =============================================================================
def _mean_sem(M):
    if M.shape[0] == 0:
        return np.full(M.shape[1], np.nan), np.full(M.shape[1], np.nan)
    n = np.sum(~np.isnan(M), axis=0)
    mean = np.nanmean(M, axis=0)
    sd = np.nanstd(M, axis=0, ddof=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        sem = sd / np.sqrt(np.maximum(n, 1))
    return mean, sem

lags_arr = np.array(LAGS)
colors = {"stim": "#1f77b4", "control": "#7f7f7f"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, outcome in zip(axes, ["reward", "no_reward"]):
    for cond in ["control", "stim"]:
        M = groups[(outcome, cond)]
        mean, sem = _mean_sem(M)
        ax.plot(lags_arr, mean, "-o", color=colors[cond], label=f"{cond} (n={M.shape[0]})")
        ax.fill_between(lags_arr, mean - sem, mean + sem, color=colors[cond], alpha=0.2)
    Ms, Mc = groups[(outcome, "stim")], groups[(outcome, "control")]
    for k, l in enumerate(LAGS):
        a = Ms[:, k]; b = Mc[:, k]
        a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
        if len(a) >= 3 and len(b) >= 3:
            _, p = mannwhitneyu(a, b, alternative="two-sided")
            if p < 0.05:
                ytxt = np.nanmax([np.nanmean(Ms[:, k]), np.nanmean(Mc[:, k])])
                ax.text(l, ytxt, "*", ha="center", va="bottom", fontsize=14)
    ax.axvline(0, color="k", ls=":", lw=0.8)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("valid-trial lag from trigger")
    ax.set_title(f"{outcome} trigger")
    ax.legend(fontsize=8)
axes[0].set_ylabel(f"d{LATENT_NAME} relative to lag {BASELINE_LAG}")
fig.suptitle(f"Latent update following stimulation - {LATENT_NAME} ({MODEL_ALIAS})")
plt.tight_layout()
fig.savefig(RESULTS_DIR / f"trajectory_{LATENT_NAME}.png", dpi=150)
plt.show()


## 7. Immediate (one-step) update: stim vs control

The update at the trigger itself is `traj(lag 0) - traj(baseline)` = `y[trigger] - y[trigger-1]`.
We compare stim vs control within each outcome (Mann-Whitney U). A difference here means the
stimulation altered the value update produced by that trial's outcome.


In [ ]:
# =============================================================================
# 7. ONE-STEP UPDATE: stim vs control within each outcome
# =============================================================================
k0 = LAGS.index(0)
upd = {key: M[:, k0] for key, M in groups.items()}
upd = {key: v[~np.isnan(v)] for key, v in upd.items()}

positions = {"reward": 0, "no_reward": 1}
offsets = {"control": -0.18, "stim": 0.18}
rng = np.random.default_rng(0)

fig, ax = plt.subplots(figsize=(6, 4))
summary_rows = []
for outcome in ["reward", "no_reward"]:
    for cond in ["control", "stim"]:
        v = upd[(outcome, cond)]
        x = positions[outcome] + offsets[cond]
        ax.scatter(x + rng.normal(0, 0.03, len(v)), v, s=10, color=colors[cond], alpha=0.4)
        if len(v):
            m = np.mean(v); s = np.std(v, ddof=1) / np.sqrt(len(v))
            ax.errorbar(x, m, yerr=s, fmt="o", color="k", capsize=4, zorder=5)
    a = upd[(outcome, "stim")]; b = upd[(outcome, "control")]
    p = mannwhitneyu(a, b, alternative="two-sided")[1] if (len(a) >= 3 and len(b) >= 3) else np.nan
    summary_rows.append(dict(
        outcome=outcome,
        control_mean=np.mean(b) if len(b) else np.nan, n_control=len(b),
        stim_mean=np.mean(a) if len(a) else np.nan, n_stim=len(a),
        diff=(np.mean(a) - np.mean(b)) if (len(a) and len(b)) else np.nan,
        mannwhitney_p=p,
    ))
    yt = ax.get_ylim()[1]
    ax.text(positions[outcome], yt, f"p={p:.2g}" if np.isfinite(p) else "n/a",
            ha="center", va="top")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks([0, 1]); ax.set_xticklabels(["reward", "no-reward"])
ax.set_ylabel(f"one-step update  d{LATENT_NAME}")
ax.set_title(f"Immediate latent update: stim vs control\n{LATENT_NAME} ({MODEL_ALIAS})")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0], [0], marker="o", ls="", color=colors[c], label=c)
                   for c in ["control", "stim"]], fontsize=8)
plt.tight_layout()
fig.savefig(RESULTS_DIR / f"onestep_update_stim_vs_control_{LATENT_NAME}.png", dpi=150)
plt.show()

upd_df = pd.DataFrame(summary_rows)
upd_df.to_csv(RESULTS_DIR / f"onestep_update_stim_vs_control_{LATENT_NAME}.csv", index=False)
display(upd_df)


## 8. Per-lag summary table

Cumulative change `y[lag] - y[baseline]` per outcome and lag, with the stim-vs-control
Mann-Whitney p-value. Saved to CSV alongside the figures.


In [ ]:
# =============================================================================
# 8. PER-LAG SUMMARY TABLE (cumulative change + stim-vs-control test)
# =============================================================================
rows_out = []
for outcome in ["reward", "no_reward"]:
    Ms, Mc = groups[(outcome, "stim")], groups[(outcome, "control")]
    for k, l in enumerate(LAGS):
        a = Ms[:, k]; b = Mc[:, k]
        a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
        p = mannwhitneyu(a, b, alternative="two-sided")[1] if (len(a) >= 3 and len(b) >= 3) else np.nan
        rows_out.append(dict(
            outcome=outcome, lag=l,
            control_mean=np.mean(b) if len(b) else np.nan, n_control=len(b),
            stim_mean=np.mean(a) if len(a) else np.nan, n_stim=len(a),
            diff=(np.mean(a) - np.mean(b)) if (len(a) and len(b)) else np.nan,
            mannwhitney_p=p,
        ))
lag_df = pd.DataFrame(rows_out)
lag_path = RESULTS_DIR / f"perlag_summary_{LATENT_NAME}.csv"
lag_df.to_csv(lag_path, index=False)
print(f"saved -> {lag_path}")
display(lag_df)


## 9. (Optional) sweep several latents

Reuse the same trigger logic for a set of latents and lay out their trajectories as a grid
(rows = latent, columns = reward/no-reward), each panel showing stim vs control.


In [ ]:
# =============================================================================
# 9. (OPTIONAL) SWEEP SEVERAL LATENTS
# =============================================================================
SWEEP_LATENTS = ["sumQ", "deltaQ", "chosenQ", "right_choice_probability", "RPE"]

def latent_series(latent_name):
    yy = extract_fitted_data(nwb, fitted_latent=fit, model_alias=MODEL_ALIAS, latent_name=latent_name)
    if yy is None:
        return None
    yy = np.asarray(yy, dtype=float)
    return yy if len(yy) == n_valid else None

fig, axes = plt.subplots(len(SWEEP_LATENTS), 2, figsize=(11, 2.8 * len(SWEEP_LATENTS)),
                         sharex=True, squeeze=False)
for row, latent_name in enumerate(SWEEP_LATENTS):
    series = latent_series(latent_name)
    gg = group_trajectories(series) if series is not None else None
    for col, outcome in enumerate(["reward", "no_reward"]):
        ax = axes[row][col]
        if gg is None:
            ax.text(0.5, 0.5, f"{latent_name}: n/a", transform=ax.transAxes, ha="center")
            ax.axis("off"); continue
        for cond in ["control", "stim"]:
            M = gg[(outcome, cond)]
            mean, sem = _mean_sem(M)
            ax.plot(lags_arr, mean, "-o", color=colors[cond], label=cond)
            ax.fill_between(lags_arr, mean - sem, mean + sem, color=colors[cond], alpha=0.2)
        ax.axvline(0, color="k", ls=":", lw=0.8); ax.axhline(0, color="k", lw=0.8)
        if row == 0:
            ax.set_title(f"{outcome} trigger")
        if col == 0:
            ax.set_ylabel(f"d{latent_name}")
        if row == len(SWEEP_LATENTS) - 1:
            ax.set_xlabel("valid-trial lag")
axes[0][0].legend(fontsize=8)
fig.suptitle(f"Latent updates following stimulation - multiple latents ({MODEL_ALIAS})")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "trajectory_multi_latent.png", dpi=150)
plt.show()
